In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 옵션 지정하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
옵션을 사용하면 Estimator와 Sampler 프리미티브를 사용자 정의할 수 있습니다. 이 섹션에서는 Qiskit Runtime 프리미티브 옵션을 지정하는 방법에 초점을 맞춥니다. 프리미티브의 `run()` 메서드 인터페이스는 모든 구현에서 공통이지만, 옵션은 그렇지 않습니다. [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) 및 [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) 옵션에 대한 자세한 내용은 해당 API 참조를 확인하세요.

프리미티브에서 옵션을 지정할 때 참고할 사항:

- `SamplerV2`와 `EstimatorV2`는 별도의 옵션 클래스를 가지고 있습니다. 프리미티브 초기화 중 또는 초기화 후에 사용 가능한 옵션을 확인하고 옵션 값을 업데이트할 수 있습니다.
- `update()` 메서드를 사용하여 `options` 속성에 변경 사항을 적용하세요.
- 옵션 값을 지정하지 않으면 특수 값인 `Unset`이 부여되고 서버 기본값이 사용됩니다.
- `options` 속성은 Python의 `dataclass` 타입입니다. 내장 `asdict` 메서드를 사용하여 딕셔너리로 변환할 수 있습니다.

<span id="pass-options"></span>
## 프리미티브 옵션 설정하기
프리미티브를 초기화할 때, 초기화 후, 또는 `run()` 메서드에서 옵션을 설정할 수 있습니다. 동일한 옵션이 여러 곳에서 지정될 때 어떻게 처리되는지 이해하려면 [우선순위 규칙](runtime-options-overview#options-precedence) 섹션을 참조하세요.

### 프리미티브 초기화
프리미티브를 초기화할 때 옵션 클래스의 인스턴스나 딕셔너리를 전달할 수 있으며, 그러면 해당 옵션의 복사본이 만들어집니다. 따라서 원본 딕셔너리나 옵션 인스턴스를 변경해도 프리미티브가 소유한 옵션에는 영향을 미치지 않습니다.

#### 옵션 클래스
`EstimatorV2` 또는 `SamplerV2` 클래스의 인스턴스를 생성할 때 옵션 클래스의 인스턴스를 전달할 수 있습니다. 그러면 `run()`을 사용하여 계산을 수행할 때 해당 옵션이 적용됩니다. `options.option.sub-option.sub-sub-option = choice` 형식으로 옵션을 지정하세요. 예: `options.dynamical_decoupling.enable = True`

예시:

`SamplerV2`와 `EstimatorV2`는 별도의 옵션 클래스([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options)와 [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options))를 가지고 있습니다.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### 딕셔너리
프리미티브를 초기화할 때 딕셔너리로 옵션을 지정할 수 있습니다.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### 초기화 후 옵션 업데이트
자동 완성 기능을 활용하려면 `primitive.options.option.sub-option.sub-sub-option = choice` 형식으로 옵션을 지정하거나, `update()` 메서드를 사용하여 일괄 업데이트할 수 있습니다.

프리미티브 초기화 후 옵션을 설정하는 경우 `SamplerV2`와 `EstimatorV2` 옵션 클래스([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options)와 [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options))를 인스턴스화할 필요가 없습니다.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Run() 메서드
`run()`에 전달할 수 있는 값은 인터페이스에 정의된 것들뿐입니다. 즉, Sampler의 경우 `shots`, Estimator의 경우 `precision`입니다. 이 값은 현재 실행에 대해 설정된 `default_shots` 또는 `default_precision` 값을 덮어씁니다.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### 특수 케이스
#### 복원력 수준 (Estimator 전용)
복원력 수준은 프리미티브 쿼리에 직접 영향을 미치는 옵션이 아니라, 기반이 되는 큐레이션된 옵션 세트를 지정합니다. 일반적으로 레벨 0은 모든 오류 완화를 끄고, 레벨 1은 측정 오류 완화 옵션을 켜며, 레벨 2는 게이트 및 측정 오류 완화 옵션을 켭니다.

복원력 수준 외에 수동으로 지정하는 옵션은 복원력 수준이 정의한 기본 옵션 세트 위에 적용됩니다. 따라서 원칙적으로 복원력 수준을 1로 설정한 후 측정 완화를 끌 수 있지만, 이는 권장되지 않습니다.

다음 예시에서는 복원력 수준을 0으로 설정하면 초기에 `zne_mitigation`이 꺼지지만, `estimator.options.resilience.zne_mitigation = True`가 `estimator.options.resilience_level = 0`의 관련 설정을 재정의합니다.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### 샷 수 (Sampler 전용)
`SamplerV2.run` 메서드는 두 가지 인수를 받습니다: PUB별 샷 수를 지정할 수 있는 PUB 목록과 shots 키워드 인수입니다. 이 샷 값들은 Sampler 실행 인터페이스의 일부로, Runtime Sampler의 옵션과는 독립적입니다. Sampler 추상화를 준수하기 위해 옵션으로 지정된 값보다 우선합니다.

그러나 어떤 PUB이나 run 키워드 인수에서도 `shots`가 지정되지 않은 경우(또는 모두 `None`인 경우), 옵션의 샷 값, 특히 `default_shots`가 사용됩니다.

정리하면, 특정 PUB에 대해 Sampler에서 샷 수를 지정하는 우선순위는 다음과 같습니다:

1. PUB에 샷 수가 지정된 경우 해당 값을 사용합니다.
2. `run`의 `shots` 키워드 인수가 지정된 경우 해당 값을 사용합니다.
3. `num_randomizations`와 `shots_per_randomization`이 `twirling` 옵션으로 지정된 경우 샷 수는 해당 값들의 곱입니다.
3. `sampler.options.default_shots`가 지정된 경우 해당 값을 사용합니다.

따라서 모든 가능한 위치에 샷 수가 지정된 경우, 가장 높은 우선순위(PUB에 지정된 샷 수)를 가진 값이 사용됩니다.

#### 정밀도 (Estimator 전용)
정밀도는 이전 섹션에서 설명한 샷 수와 유사하지만, Estimator 옵션에는 `default_shots`와 `default_precision`이 모두 포함된다는 점이 다릅니다. 또한 게이트 트월링이 기본적으로 활성화되어 있으므로, `num_randomizations`와 `shots_per_randomization`의 곱이 이 두 옵션보다 우선합니다.

구체적으로, 특정 Estimator PUB에 대한 우선순위는 다음과 같습니다:

1. PUB에 정밀도가 지정된 경우 해당 값을 사용합니다.
2. `run`의 precision 키워드 인수가 지정된 경우 해당 값을 사용합니다.
2. `num_randomizations`와 `shots_per_randomization`이 [`twirling` 옵션](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)으로 지정된 경우(기본적으로 활성화), 해당 값들의 곱을 사용하여 데이터 양을 제어합니다.
3. `estimator.options.default_shots`가 지정된 경우 해당 값을 사용하여 데이터 양을 제어합니다.
4. `estimator.options.default_precision`이 지정된 경우 해당 값을 사용합니다.

예를 들어, 네 곳 모두에 정밀도가 지정된 경우 가장 높은 우선순위(PUB에 지정된 정밀도)를 가진 값이 사용됩니다.

> **Note:** 정밀도는 사용량과 반비례 관계입니다. 즉, 정밀도가 낮을수록 실행에 더 많은 QPU 시간이 소요됩니다.

## 자주 사용되는 옵션
사용 가능한 옵션은 많지만, 다음이 가장 자주 사용됩니다:

<span id="shots"></span>
### 샷 수
일부 알고리즘에서는 특정 샷 수를 설정하는 것이 루틴의 핵심 부분입니다. 샷 수(또는 정밀도)는 여러 위치에서 지정할 수 있으며, 우선순위는 다음과 같습니다:

Sampler PUB의 경우:

1. PUB에 포함된 정수 값의 샷 수
2. `run(...,shots=val)` 값
3. `options.default_shots` 값

Estimator PUB의 경우:

1. PUB에 포함된 부동 소수점 값의 정밀도
2. `run(...,precision=val)` 값
3. `options.default_shots` 값
4. `options.default_precision` 값

예시:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### 최대 실행 시간
최대 실행 시간(`max_execution_time`)은 작업이 실행될 수 있는 최대 시간을 제한합니다. 작업이 이 시간 제한을 초과하면 강제로 취소됩니다. 이 값은 작업, Session, 또는 배치 모드로 실행되는 단일 작업에 적용됩니다.

값은 벽시계 시간이 아닌 양자 시간(QPU가 작업 처리에 전용으로 사용하는 시간) 기준으로 초 단위로 설정됩니다. 로컬 테스트 모드는 양자 시간을 사용하지 않으므로 이 모드에서는 무시됩니다.